# Amaliy mashgʻulot 11. Matnlar oʻxshashligi: kosinus oʻxshashligi, Jaccard indeksi va Levenshtein masofasi

11-laboratoriyada Levenshtein jadvalini katakma-katak toʻldirdingiz, Jaccard uchun Venn diagrammasini va TF-IDF vektorlari orasidagi burchakni koʻrdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. Levenshtein masofasini dinamik dasturlash bilan yozamiz, jadvalni chop etamiz va tahrirlar zanjirini tiklaymiz; `nltk.edit_distance` bilan solishtiramiz;
2. Jaccard indeksini soʻzlar, oʻzaklar va belgili n-grammlar boʻyicha hisoblaymiz va qoʻshimchalar soʻzlar boʻyicha solishtirishni qanday «buzishini» koʻramiz;
3. TF-IDF ni qoʻlda quramiz, kosinusni alohida soʻzlar hissasiga ajratamiz va `scikit-learn` bilan tekshiramiz;
4. uchala oʻlchovni olti xil juft gapda solishtiramiz va ikki masalaga qoʻllaymiz: imlo xatolarini tuzatish va deyarli takroriy matnlarni topish.

*«NLP algoritmlari» kursining 11-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Levenshtein masofasi

Levenshtein masofasi — $a$ satrni $b$ satrga aylantiradigan eng kam tahrirlar soni (belgi qoʻshish, oʻchirish, almashtirish). U dinamik dasturlash bilan hisoblanadi: $D[i][j]$ — $a$ ning dastlabki $i$ ta belgisi va $b$ ning dastlabki $j$ ta belgisi orasidagi masofa:

$$D[i][j] = \min\big(D[i-1][j] + c_{del},\; D[i][j-1] + c_{ins},\; D[i-1][j-1] + c_{sub}\cdot[a_i \ne b_j]\big)$$

Birinchi satr va birinchi ustun — boʻsh satr ε gacha boʻlgan masofa. Javob pastki oʻng burchakda turadi. Tahrirlarning oʻzini bilish uchun burchakdan orqaga yuramiz (backtrace): har safar minimum kelgan katakni tanlaymiz. Misol laboratoriyadagi bilan bir xil: «kitob» → «kitoblar».

In [ ]:
import numpy as np

def levenshtein_table(a, b, sub_cost=1, ins_cost=1, del_cost=1):
    # D[i, j] = a[:i] va b[:j] orasidagi masofa
    n, m = len(a), len(b)
    D = np.zeros((n + 1, m + 1))
    D[:, 0] = np.arange(n + 1) * del_cost   # a[:i] -> boʻsh: faqat oʻchirish
    D[0, :] = np.arange(m + 1) * ins_cost   # boʻsh -> b[:j]: faqat qoʻshish
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = D[i - 1, j - 1] + (0 if a[i - 1] == b[j - 1] else sub_cost)
            D[i, j] = min(D[i - 1, j] + del_cost,   # a[i-1] ni oʻchirish
                          D[i, j - 1] + ins_cost,   # b[j-1] ni qoʻshish
                          diag)                     # moslik yoki almashtirish
    return D

def print_table(a, b, D):
    print("     " + " ".join(f"{c:>3}" for c in "ε" + b))
    for i, row in enumerate(D):
        label = "ε" if i == 0 else a[i - 1]
        print(f"{label:>3}  " + " ".join(f"{v:>3g}" for v in row))

def backtrace(a, b, D, sub_cost=1, ins_cost=1, del_cost=1):
    # pastki oʻng burchakdan orqaga yurib, amallarni yozib boramiz
    i, j = len(a), len(b)
    ops, path = [], [(i, j)]
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (0 if a[i - 1] == b[j - 1] else sub_cost):
            ops.append(("=" if a[i - 1] == b[j - 1] else "sub", a[i - 1], b[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + del_cost:
            ops.append(("del", a[i - 1], "-"))
            i -= 1
        else:
            ops.append(("ins", "-", b[j - 1]))
            j -= 1
        path.append((i, j))
    return ops[::-1], path[::-1]

OP_NAMES = {"sub": "almashtirish", "del": "oʻchirish", "ins": "qoʻshish"}

def show_alignment(a, b, **costs):
    D = levenshtein_table(a, b, **costs)
    ops, _ = backtrace(a, b, D, **costs)
    print(" ".join(x for _, x, _ in ops))
    print(" ".join({"=": "|", "sub": "*"}.get(op, " ") for op, _, _ in ops))
    print(" ".join(y for _, _, y in ops))
    edits = [f"{OP_NAMES[op]} {x}→{y}" if op == "sub" else f"{OP_NAMES[op]} «{x if op == 'del' else y}»"
             for op, x, y in ops if op != "="]
    print(f"d({a}, {b}) = {D[-1, -1]:g}:", "; ".join(edits))

a, b = "kitob", "kitoblar"
D = levenshtein_table(a, b)
print_table(a, b, D)
print()
show_alignment(a, b)

Yana bir misol va kutubxona bilan tekshirish. `nltk.edit_distance` xuddi shuni hisoblaydi; `transpositions=True` toʻrtinchi amalni — qoʻshni harflarning oʻrnini almashtirishni qoʻshadi (Damerau–Levenshtein masofasi), `substitution_cost=2` esa almashtirishni qimmatroq qiladi (bunda almashtirish = oʻchirish + qoʻshish).

Eʼtibor bering: oʻzbek lotin yozuvida «oʻ», «gʻ» ikki belgidan iborat (`o` + `ʻ`), shuning uchun «oʻ» → «u» xatosi 2 ta tahrir hisoblanadi, tushib qolgan tutuq belgisi (`ʼ`) yoki `ʻ` esa 1 ta.


Katak oxirida — jadval rasm koʻrinishida: rang — $D[i][j]$ qiymati, toʻq sariq chiziq — backtrace yoʻli. Diagonal qadam — moslik yoki almashtirish, gorizontal — qoʻshish, vertikal — oʻchirish.

In [ ]:
import nltk

show_alignment("oʻqituvchi", "uqituvchi")
print()

pairs_lev = [("kitob", "kitoblar"), ("rahmat", "rhamat"), ("maʼruza", "maruza"),
             ("lugʻat", "lugat"), ("havo", "xavo"), ("oʻqituvchi", "uqituvchi")]
print(f"{'':<26}{'bizniki':>8}{'nltk':>6}{'+oʻrin':>8}{'sub=2':>7}")
for a, b in pairs_lev:
    mine = levenshtein_table(a, b)[-1, -1]
    ref = nltk.edit_distance(a, b)
    dl = nltk.edit_distance(a, b, transpositions=True)
    sub2 = levenshtein_table(a, b, sub_cost=2)[-1, -1]
    assert mine == ref and sub2 == nltk.edit_distance(a, b, substitution_cost=2)
    print(f"{a + ' / ' + b:<26}{mine:>8g}{ref:>6}{dl:>8}{sub2:>7g}")

import matplotlib.pyplot as plt

a, b = "kitob", "kitoblar"
D = levenshtein_table(a, b)
_, path = backtrace(a, b, D)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(D, cmap="Blues")
for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        ax.text(j, i, f"{D[i, j]:g}", ha="center", va="center",
                color="white" if D[i, j] > 0.6 * D.max() else "black")
pi, pj = zip(*path)
ax.plot(pj, pi, color="#D9480F", lw=2.5, marker="o")
ax.set_xticks(range(len(b) + 1))
ax.set_xticklabels(["ε"] + list(b))
ax.set_yticks(range(len(a) + 1))
ax.set_yticklabels(["ε"] + list(a))
ax.xaxis.tick_top()
ax.set_title("Levenshtein jadvali va backtrace yoʻli")
plt.show()

## 2. Jaccard indeksi: soʻzlar, oʻzaklar, belgili n-grammlar

Jaccard indeksi ikki **toʻplamni** solishtiradi: $J(A,B) = \dfrac{|A \cap B|}{|A \cup B|}$. Bir xil toʻplamlar uchun u 1 ga, kesishmaydiganlar uchun 0 ga teng. Toʻplam elementlari nima boʻlishini oʻzimiz tanlaymiz: soʻzlar, soʻz oʻzaklari yoki belgili n-grammlar ($n$ harfli boʻlakchalar; `#` soʻz chegarasini bildiradi).

Maʼnosi deyarli bir xil, lekin soʻz shakllari har xil boʻlgan ikki gapni olamiz.


Soʻngra normallashtirish yordam berishini tekshiramiz. Lekin `nltk` dagi Snowball stemmerlari orasida oʻzbek tili yoʻq, tayyor oʻzbek stemmerlari ham kam. Shuning uchun oddiy qoidaviy stemmer yozamiz: soʻz oxiridan kelishik, egalik va koʻplik qoʻshimchalarini (shu tartibda, har biridan bittadan) olib tashlaymiz. Bu qoʻpol usul: masalan, «kursi» soʻzida `-si` ni egalik qoʻshimchasi deb xato kesib tashlaydi.

In [ ]:
import re
from nltk.metrics.distance import jaccard_distance

def tokens(text):
    return re.findall(r"[а-яёa-zʻʼ'-]+", text.lower())

def jaccard(A, B):
    A, B = set(A), set(B)
    return len(A & B) / len(A | B) if (A | B) else 1.0

def char_ngrams(text, n=3):
    # har bir soʻzning # chegarali belgili n-grammlari
    grams = set()
    for w in tokens(text):
        w = "#" + w + "#"
        grams |= {w[i:i + n] for i in range(len(w) - n + 1)}
    return grams

s1 = "Talabalar kutubxonada yangi kitoblarni oʻqidi"
s2 = "Talaba kutubxonadagi yangi kitobni oʻqidi"
A, B = set(tokens(s1)), set(tokens(s2))
print("A ∩ B =", sorted(A & B))
print("A ∪ B =", sorted(A | B))
print(f"soʻzlar boʻyicha Jaccard: {len(A & B)}/{len(A | B)} = {jaccard(A, B):.3f}")
print(f"nltk:  1 - jaccard_distance = {1 - jaccard_distance(A, B):.3f}")
print()
for n in (2, 3, 4):
    GA, GB = char_ngrams(s1, n), char_ngrams(s2, n)
    print(f"n-grammlar boʻyicha Jaccard n={n}: {len(GA & GB):>2}/{len(GA | GB):<3} = {jaccard(GA, GB):.3f}")
print("umumiy trigrammlar (dastlabki 12 tasi):", sorted(char_ngrams(s1, 3) & char_ngrams(s2, 3))[:12])

# qoʻshimchalar guruhlari soʻz oxiridan boshlab: kelishik -> egalik -> koʻplik
CASE = ["ning", "dagi", "dan", "da", "ga", "ka", "qa", "ni"]
POSS = ["imiz", "ingiz", "lari", "im", "ing", "si"]
PLUR = ["lar"]

def stem(word, min_len=3):
    for group in (CASE, POSS, PLUR):
        for suf in group:                      # har bir guruhdan koʻpi bilan bitta qoʻshimcha
            if word.endswith(suf) and len(word) - len(suf) >= min_len:
                word = word[: -len(suf)]
                break
    return word

for w1, w2 in zip(tokens(s1), tokens(s2)):
    print(f"{w1:<14} → {stem(w1):<11} {w2:<14} → {stem(w2)}")
SA, SB = {stem(w) for w in tokens(s1)}, {stem(w) for w in tokens(s2)}
print(f"\noʻzaklar boʻyicha Jaccard: {len(SA & SB)}/{len(SA | SB)} = {jaccard(SA, SB):.3f}")

Natija $n$ ga qanday bogʻliq? Solishtirish uchun ruscha juftni ham qoʻshamiz. Oʻzbek tilida bitta soʻz son, egalik va kelishikni birga tashiydi («kitob-lar-ni»), shuning uchun grammatikaning har qanday oʻzgarishi butun soʻzni satr sifatida oʻzgartiradi. Soʻzlar boʻyicha moslik kam, belgili n-grammlar esa umumiy oʻzaklarni saqlab qoladi. $n=1$ da deyarli barcha harflar umumiy (oʻxshashlik oshirib koʻrsatiladi), katta $n$ da n-grammlar «deyarli soʻzga» aylanadi.

In [ ]:
import matplotlib.pyplot as plt

other = ("Студенты читают новые книги в библиотеке", "Студент читает новую книгу в библиотеке")   # xuddi shu maʼnodagi ruscha juft
examples = {"oʻzbekcha juft": (s1, s2), "ruscha juft": other}
ns = range(1, 7)
fig, ax = plt.subplots(figsize=(7, 3.5))
for (label, (x, y)), color in zip(examples.items(), ["#2B55C9", "#17907E"]):
    ax.plot(ns, [jaccard(char_ngrams(x, n), char_ngrams(y, n)) for n in ns], marker="o", color=color,
            label=label + ": n-grammlar")
    ax.axhline(jaccard(tokens(x), tokens(y)), ls="--", color=color, label=label + ": soʻzlar")
ax.set_xlabel("n")
ax.set_ylabel("Jaccard indeksi")
ax.set_ylim(0, 1)
ax.set_title("Belgili n-grammlar va soʻzlar boʻyicha Jaccard")
ax.legend(fontsize=8)
plt.show()

## 3. Kosinus oʻxshashligi va TF-IDF

Matnni vektorga aylantiramiz: lugʻatdagi har bir soʻzga bitta oʻq, qiymati — TF-IDF vazni: $\text{tf}(t,d)\cdot \text{idf}(t)$, bu yerda $\text{idf}(t) = \ln\frac{1+N}{1+\text{df}(t)} + 1$ (`scikit-learn` shunday hisoblaydi; $N$ — hujjatlar soni, df — soʻz nechta hujjatda uchrashi). Kam uchraydigan soʻzlar kattaroq vazn oladi. Vektorlar orasidagi burchak kosinusi:

$$\cos(\vec a, \vec b) = \frac{\sum_i a_i b_i}{\lVert \vec a\rVert\, \lVert \vec b\rVert}$$

Quyida olti juft gap: oʻlchovlar ularda turlicha ishlaydi (laboratoriyadagi jadvaldagidek). TF-IDF ni qoʻlda hisoblaymiz va «inkor» jufti uchun kosinusni soʻzlar hissasiga ajratamiz.


Katak oxirida natijani `scikit-learn` bilan tekshiramiz: `TfidfVectorizer` (bizning tokenizator bilan) va `cosine_similarity`. Sonlar mos kelishi kerak.

In [ ]:
pairs = [
    ("soʻz tartibi",    "Talaba qiyin masalani tez yechdi", "Qiyin masalani talaba tez yechdi"),
    ("imlo xatolari",   "Ertaga Samarqandda havo iliq boʻladi", "Ertaga Samarqanda xavo iliq boʻladi"),
    ("soʻz oʻzgarishi", "Talabalar kutubxonada yangi kitoblarni oʻqidi", "Talaba kutubxonadagi yangi kitobni oʻqidi"),
    ("sinonimlar",      "Shifokor koʻproq sayr qilishni maslahat berdi", "Doktor tez-tez ochiq havoda yurishni tavsiya qildi"),
    ("inkor",           "Film menga juda yoqdi", "Film menga umuman yoqmadi"),
    ("boshqa mavzu",    "Birjada paxta narxi oshdi", "Futbol oʻyini shanbaga koʻchirildi"),
]

from collections import Counter

docs = [s for _, a, b in pairs for s in (a, b)]
tok_docs = [tokens(d) for d in docs]
vocab = sorted({w for d in tok_docs for w in d})
N = len(docs)
df = Counter(w for d in tok_docs for w in set(d))
# scikit-learn dagi kabi silliqlangan idf
idf = {w: np.log((1 + N) / (1 + df[w])) + 1 for w in vocab}

def tfidf_vector(toks):
    tf = Counter(toks)
    v = np.array([tf[w] * idf[w] for w in vocab])
    return v / np.linalg.norm(v)          # L2-meʼyorlash: vektor uzunligi = 1

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

k = 4                                     # «inkor» jufti
name, a, b = pairs[k]
va, vb = tfidf_vector(tokens(a)), tfidf_vector(tokens(b))
print(a, "|", b)
print(f"   {'soʻz':<12} {'idf':>5}   a_i × b_i")
for i, w in enumerate(vocab):
    if va[i] * vb[i] > 0:
        print(f"   {w:<12} {idf[w]:5.2f}   {va[i]:.3f} × {vb[i]:.3f} = {va[i] * vb[i]:.3f}")
print(f"cos = {cosine(va, vb):.3f}")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(tokenizer=tokens, token_pattern=None, lowercase=False)
X = vec.fit_transform(docs)
assert list(vec.get_feature_names_out()) == vocab
assert np.allclose(X.toarray()[2 * k], va)        # vektorlar biznikiga mos keladi
S = cosine_similarity(X)
print(f"sklearn: {S[2 * k, 2 * k + 1]:.3f}   qoʻlda: {cosine(va, vb):.3f}")

Endi barcha oʻlchovlarni olti juftda solishtiramiz. Lev — butun satrlar boʻyicha meʼyorlangan Levenshtein oʻxshashligi $1 - d/\max(|a|,|b|)$; J — Jaccard; cos — soʻzlar va belgili n-grammlar ($n$ = 2…4) boʻyicha TF-IDF kosinusi.

In [ ]:
import pandas as pd

char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))   # soʻz ichidagi belgili n-grammlar, n = 2..4
Sc = cosine_similarity(char_vec.fit_transform(docs))

rows = []
for k, (name, a, b) in enumerate(pairs):
    d = nltk.edit_distance(a.lower(), b.lower())
    rows.append({
        "juft": name,
        "Lev": 1 - d / max(len(a), len(b)),
        "J soʻzlar": jaccard(tokens(a), tokens(b)),
        "J 3-gramm": jaccard(char_ngrams(a, 3), char_ngrams(b, 3)),
        "cos soʻzlar": S[2 * k, 2 * k + 1],
        "cos belgilar": Sc[2 * k, 2 * k + 1],
    })
table = pd.DataFrame(rows).set_index("juft").round(2)
print(table.to_string())

Jadvaldan nimalar koʻrinadi:

- **soʻz tartibi**: soʻzlar qopi tartibni koʻrmaydi (soʻzlar boʻyicha J va cos = 1), satr boʻyicha Levenshtein esa sezilarli pasayadi;
- **imlo xatolari** va **soʻz oʻzgarishi**: soʻzlar boʻyicha oʻlchovlar pasayadi, belgili oʻlchovlar yuqori qoladi;
- **sinonimlar**: maʼno bir xil boʻlsa ham, soʻzlar boʻyicha oʻlchovlar nolga teng, belgilar boʻyicha esa juda past (0.1–0.2);
- **inkor**: maʼno qarama-qarshi boʻlsa ham, oʻxshashlik ancha yuqori (ayniqsa Levenshtein boʻyicha);
- **boshqa mavzu**: soʻzlar boʻyicha nol, belgilar boʻyicha esa umumiy harflar kichik «shovqin» beradi; Levenshtein hatto bogʻliq boʻlmagan satrlar uchun ham 0.2–0.3 atrofida boʻladi.

### Haqiqiy modelda

Neyron tarmoq asosidagi gap embeddinglari harflarni emas, maʼnoni solishtiradi. `intfloat/multilingual-e5-small` modeli (≈470 MB, rus va oʻzbek tillarini qoʻllab-quvvatlaydi) gapni vektorga aylantiradi; keyin xuddi shu kosinus. Model mualliflari matn oldiga `"query: "` prefiksini qoʻshishni maslahat beradi va bu modelda kosinuslar odatda 0.7–1.0 oraligʻida boʻlishini ogohlantiradi, shuning uchun 0.5 chegarasi bilan emas, juftlarni bir-biri bilan solishtirish muhim. «Sinonimlar» jufti koʻtariladimi va model «inkor»ni farqlaydimi — tekshiring. Kataklar faqat Colab da ishlaydi (internet kerak).

In [ ]:
!pip install -q sentence-transformers

# gap embeddinglari; model bir marta yuklab olinadi (≈470 MB)
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("intfloat/multilingual-e5-small")
    for name, a, b in pairs:
        emb = model.encode(["query: " + a, "query: " + b], normalize_embeddings=True)
        print(f"{name:<18} cos = {float(emb[0] @ emb[1]):.3f}")
except Exception as e:
    print("Modelni yuklab boʻlmadi (internet va sentence-transformers paketi kerak):", e)

## 4. Qoʻllanish: imlo xatolarini tuzatish

Eng oddiy tuzatuvchi: lugʻatda yoʻq soʻz uchun lugʻatdan Levenshtein masofasi eng kichik soʻzlarni olamiz. Quyida 40 soʻzli lugʻat (bosh shaklda), xatoli soʻzlar va qoʻshimchali toʻgʻri soʻzlar. Oxirgilarini ham tuzatuvchi «tuzatadi», chunki ular lugʻatda yoʻq. Koʻp uchraydigan xatolarga eʼtibor bering: tushib qolgan `ʼ` («maruza») va `ʻ` («lugat»).

In [ ]:
vocab = ["universitet", "talaba", "oʻqituvchi", "maʼruza", "amaliyot", "laboratoriya", "kutubxona", "kitob",
         "darslik", "imtihon", "topshiriq", "savol", "javob", "tarjima", "lugʻat", "gap", "soʻz", "matn", "til",
         "tilshunoslik", "algoritm", "dastur", "kompyuter", "maʼlumot", "model", "masofa", "oʻxshashlik", "vektor",
         "matritsa", "shahar", "maktab", "daftar", "ruchka", "jadval", "harf", "belgi", "satr", "qoida", "misol", "natija"]
typos = ["univeristet", "kutubxna", "tarjma", "algaritm", "oʻxshashlk", "maruza", "lugat"]
inflected = ["talabalarga", "kitoblarimizdan", "kutubxonda"]   # qoʻshimchali soʻzlar (oxirgisida xato ham bor)

def suggest(word, vocab, k=3):
    # Levenshtein masofasi boʻyicha lugʻatdagi eng yaxshi k ta soʻz
    return sorted((nltk.edit_distance(word, v), v) for v in vocab)[:k]

for w in typos + inflected:
    print(f"{w:<16}", ", ".join(f"{v} ({d})" for d, v in suggest(w, vocab)))

«kitoblarimizdan» shakli uchun eng yaqin soʻzlar faqat 10 va undan katta masofada: Levenshtein morfologiyani bilmaydi, oʻzbek tilida esa bitta oʻzakka koʻp qoʻshimcha qoʻshiladi. Yechim — **oʻzaklarni** solishtirish: avval stemmer, keyin lugʻatdagi oʻzaklargacha masofa. Bu xato va qoʻshimcha bir vaqtda boʻlganda ham ishlaydi («kutubxonda» → «kutubxon» → «kutubxona»).

In [ ]:
stem_vocab = {stem(v): v for v in vocab}

def suggest_stem(word, k=3):
    s = stem(word)
    return s, sorted((nltk.edit_distance(s, sv), v) for sv, v in stem_vocab.items())[:k]

for w in inflected:
    s, cands = suggest_stem(w)
    print(f"{w:<16} → {s:<12}", ", ".join(f"{v} ({d})" for d, v in cands))

## 5. Qoʻllanish: deyarli takroriy matnlarni topish

Yangiliklar agregatorlari bitta voqea haqidagi xabarlarni birlashtiradi. Quyida 10 ta sarlavha: bitta voqeaning qayta aytilgan uch guruhi (soʻz tartibi, boshqa qoʻshimcha, imlo xatosi bilan) va boshqa sarlavhalar. Soʻzlar va belgili n-grammlar boʻyicha kosinusi ≥ 0.5 boʻlgan juftlarni qidiramiz.

In [ ]:
heads = [
    "Samarqandda yangi kutubxona ochildi",                         # 0
    "Samarqandda yangi kutubxonani ochishdi",                      # 1
    "Yangi kutubxona Samarqandda ochildi",                         # 2
    "Toshkentda xalqaro marafon boʻlib oʻtdi",                      # 3
    "Xalqaro marafon Toshkentda boʻlib oʻtdi",                      # 4
    "Toshkentda xalqaro marofon boʻlib oʻtdi",                      # 5
    "Talabalar matnni qayta ishlash algoritmlarini oʻrganmoqda",    # 6
    "Talaba matnlarni qayta ishlash algoritmini oʻrgandi",          # 7
    "Ertaga Buxoroda havo issiq boʻladi",                          # 8
    "Birjada paxta narxi oshdi",                                   # 9
]
groups = [{0, 1, 2}, {3, 4, 5}, {6, 7}]   # bitta guruh ichidagi juftlar — takroriy

word_vec = TfidfVectorizer(tokenizer=tokens, token_pattern=None, lowercase=False)
S_word = cosine_similarity(word_vec.fit_transform(heads))
S_char = cosine_similarity(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(heads))

gold = {(i, j) for g in groups for i in g for j in g if i < j}
threshold = 0.5
for name, S_ in [("soʻzlar boʻyicha", S_word), ("belgili n-grammlar boʻyicha", S_char)]:
    found = {(i, j) for i in range(len(heads)) for j in range(i + 1, len(heads)) if S_[i, j] >= threshold}
    tp = len(found & gold)
    print(f"=== {name}: topilgan juftlar {len(found)}, toʻgʻrisi {tp} / {len(gold)}")
    for i, j in sorted(gold | found):
        mark = "✓" if (i, j) in found & gold else ("oʻtkazildi" if (i, j) in gold else "notoʻgʻri")
        print(f"   {S_[i, j]:.2f} {mark:<10} {heads[i]}  ~  {heads[j]}")
    print()

Belgili n-grammlar soʻzlari boshqa shaklda yoki xato bilan yozilgan qayta aytishlarni topadi. Quyidagi issiqlik xaritalarida bu diagonal boʻylab yorugʻroq bloklar sifatida koʻrinadi.

*Maslahat:* chap tomondagi «Files» paneli orqali oʻzingizning `texts.csv` faylingizni yuklab, undagi takroriy matnlarni topishingiz mumkin (`pd.read_csv("texts.csv")`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, S_, title in [(axes[0], S_word, "Soʻzlar boʻyicha TF-IDF kosinusi"), (axes[1], S_char, "Belgili n-grammlar boʻyicha TF-IDF kosinusi")]:
    im = ax.imshow(S_, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(heads)))
    ax.set_yticks(range(len(heads)))
    ax.set_title(title)
    for i in range(len(heads)):
        for j in range(len(heads)):
            if i != j and S_[i, j] >= 0.3:
                ax.text(j, i, f"{S_[i, j]:.1f}", ha="center", va="center", fontsize=7,
                        color="black" if S_[i, j] > 0.6 else "white")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

## 6. Topshiriqlar

1. **Vaznli Levenshtein.** `levenshtein_table` ni shunday oʻzgartiringki, «oʻxshash» harflarni almashtirish (h/x, a/o, o/u) 0.5 turadi. «havo/xavo» va «marafon/marofon» masofalarini hisoblang va oddiy masofa bilan solishtiring.
2. **Damerau–Levenshtein.** Dinamikaga toʻrtinchi variantni qoʻshing — qoʻshni ikki harfning oʻrnini almashtirish, narxi 1: agar `a[i-1] == b[j-2]` va `a[i-2] == b[j-1]` boʻlsa, `D[i-2][j-2] + 1`. Funksiyani `damerau(a, b)` deb nomlang (son qaytaradi); oʻz-oʻzini tekshirish topshiriq katagining pastida turibdi.
3. 3-boʻlimdagi «soʻz oʻzgarishi» va «boshqa mavzu» juftlari uchun belgili n-grammlar boʻyicha Jaccardni $n = 1…6$ da hisoblang. Qaysi $n$ da juftlar orasidagi farq eng katta?
4. 4-boʻlimdagi tuzatuvchini tezlashtiring: soʻzni faqat uzunligi 2 tadan koʻp farq qilmaydigan lugʻat soʻzlari bilan solishtiring. `typos` roʻyxatida `edit_distance` ning nechta chaqiruvi tejaldi? Takliflar oʻzgardimi?
5. Takroriy matnlarni qidirishda (5-boʻlim) chegarani 0.1 dan 0.9 gacha 0.05 qadam bilan `S_word` va `S_char` uchun alohida oʻzgartiring va F1-oʻlchovining ($F_1 = 2PR/(P+R)$) chegaraga bogʻliqlik grafigini chizing. Qaysi chegaralar oraligʻida har bir variant barcha takroriylarni xatosiz topadi?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing



# Oʻz-oʻzini tekshirish: yuqorida damerau ni yozgandan keyin katakni ishga tushiring
try:
    for a, b in [("rahmat", "rhamat"), ("kitob", "kitoblar"), ("olma", "loma"), ("matn", "matnlar")]:
        assert damerau(a, b) == nltk.edit_distance(a, b, transpositions=True), (a, b)
    print("2-topshiriq: hammasi toʻgʻri!")
except NameError:
    print("Avval 2-topshiriq katagida damerau(a, b) funksiyasini aniqlang.")

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing